# 🔬 Fine-Tuning ResNet-50 on MILK10k Skin Lesion Dataset
### Model Architecture sourced from: [Sudhandar/ResNet-50-model](https://github.com/Sudhandar/ResNet-50-model)

---

### 🎯 Key Highlights:
1. **Model Source**: Exact ResNet-50 architecture from `https://github.com/Sudhandar/ResNet-50-model` (`identity_block`, `convolutional_block`, 5-stage residual structure).
2. **Modernized for TF 2.x**: Compatible with TensorFlow 2.x and GPU acceleration.
3. **ImageNet Pretrained Weight Transfer**: Layer-by-layer weight transfer directly into Sudhandar's custom layers so genuine fine-tuning / transfer learning is performed.
4. **MILK10k Dataset Pipeline**: Dynamic class detection, Black-Hat hair removal, and data augmentation.
5. **Two-Stage Fine-Tuning**:
   - **Phase 1**: Head Warmup (Backbone frozen, Adam lr = $0.001$, 5 epochs).
   - **Phase 2**: Deep Fine-Tuning (Residual blocks unfrozen, Adam lr = $0.0001$, 25 epochs with ReduceLROnPlateau & EarlyStopping).
6. **Rich Visual Output in Every Step**: Distribution bar charts, hair-removal previews, batch grids, loss curves, confusion matrices, and diagnostic predictions.

## ⚙️ Step 0: Check GPU & Install Dependencies

In [ ]:
# Check GPU availability gracefully
import subprocess, tensorflow as tf

print("=" * 65)
print("🔍 Checking Hardware Accelerator (GPU/CPU)...\n")
try:
    smi_out = subprocess.check_output('nvidia-smi', shell=True).decode()
    print(smi_out)
except Exception:
    print("⚠️ NVIDIA GPU not detected! Colab is currently running on CPU.")
    print("👉 Switch to GPU: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.")

gpus = tf.config.list_physical_devices('GPU')
print("\n✓ TensorFlow Detected GPUs:", gpus)
if not gpus:
    print("⚠️ WARNING: Training ResNet-50 on CPU will be slow. Please enable T4 GPU above.")
print("=" * 65)

# Install dependencies
!pip install -q kaggle opencv-python-headless scikit-learn matplotlib seaborn tqdm
print("\n✓ All required dependencies successfully installed!")

## 📥 Step 1: Download MILK10k Dataset

In [ ]:
import os, glob
from pathlib import Path

# Configure Kaggle credentials for direct download
os.environ['KAGGLE_USERNAME'] = "tamal247"
os.environ['KAGGLE_KEY']      = "c8446b8c8bd05c1d059d7bab184de6d2"

DATA_ROOT = "/content/milk10k"
OUT_DIR = "/content/output"
os.environ["MILK10K_ROOT"] = DATA_ROOT
os.environ["MILK10K_OUT"]  = OUT_DIR
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Check if already downloaded
existing_csv = glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True)
if not existing_csv:
    print("⏳ Downloading MILK10k via Kaggle API...")
    !kaggle datasets download -d nguyenphucduyloc/milk10k-isic-challenge-2025 --unzip -p /content/milk10k
else:
    print("✓ Dataset already downloaded in:", DATA_ROOT)

# Print summary of downloaded files
all_files = glob.glob(f"{DATA_ROOT}/**/*", recursive=True)
imgs = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
csvs = [f for f in all_files if f.lower().endswith('.csv')]
print(f"\n📁 Storage Location : {DATA_ROOT}")
print(f"🖼️ Image Files Found : {len(imgs):,}")
print(f"📄 CSV Files Found   : {len(csvs)} -> {[Path(c).name for c in csvs]}")
print("✓ Step 1 Complete: Dataset ready!")

## 📊 Step 2: Load Ground Truth, Metadata & Class Distribution Plot

In [ ]:
import glob, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Locate GroundTruth and Metadata
gt_files = glob.glob(f"{DATA_ROOT}/**/*GroundTruth*.csv", recursive=True)
if not gt_files: gt_files = glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True)
gt_path = next((p for p in gt_files if "train" in Path(p).name.lower()), gt_files[0])

meta_files = glob.glob(f"{DATA_ROOT}/**/*Metadata*.csv", recursive=True)
meta_path = next((p for p in meta_files if "train" in Path(p).name.lower()), (meta_files[0] if meta_files else None))

print("=" * 65)
print(f"✓ Ground Truth CSV: {gt_path}")
print(f"✓ Metadata CSV    : {meta_path}")
print("=" * 65)

gt = pd.read_csv(gt_path)
id_col = gt.columns[0]
class_cols = [c for c in gt.columns[1:] if c.lower() not in ("unnamed: 0", "index")]

if len(class_cols) > 1 and np.issubdtype(gt[class_cols[0]].dtype, np.number):
    gt["label"] = gt[class_cols].values.argmax(axis=1)
    classes = class_cols
else:
    label_col = class_cols[0]
    classes = sorted(gt[label_col].unique().tolist())
    gt["label"] = gt[label_col].map({c: i for i, c in enumerate(classes)})

# 2. Scan images and map lesion IDs
exts = (".jpg", ".jpeg", ".png")
all_imgs = [p for p in glob.glob(f"{DATA_ROOT}/**/*", recursive=True) if p.lower().endswith(exts)]
train_imgs = [p for p in all_imgs if "train" in p.lower()] or all_imgs

df_imgs = pd.DataFrame({"path": train_imgs})
df_imgs["stem"] = df_imgs["path"].apply(lambda p: Path(p).stem)
df_imgs["parent_folder"] = df_imgs["path"].apply(lambda p: Path(p).parent.name)
df_imgs = df_imgs.drop_duplicates(subset=["stem"]).reset_index(drop=True)

if meta_path:
    meta = pd.read_csv(meta_path)
    low_cols = {c.lower(): c for c in meta.columns}
    stem_col = next((low_cols[c] for c in ("isic_id", "image_id", "image", "filename") if c in low_cols), None)
    lesion_col = next((low_cols[c] for c in ("lesion_id", "lesion", "patient_id") if c in low_cols), None)
    if stem_col and lesion_col:
        meta["_stem"] = meta[stem_col].astype(str).apply(lambda s: Path(s).stem)
        l_map = dict(zip(meta["_stem"], meta[lesion_col].astype(str)))
        df_imgs["lesion"] = df_imgs["stem"].map(l_map).fillna(df_imgs["parent_folder"])
    else:
        df_imgs["lesion"] = df_imgs["stem"]
else:
    df_imgs["lesion"] = df_imgs["stem"]

gt[id_col] = gt[id_col].astype(str)
if set(df_imgs["lesion"]).intersection(set(gt[id_col])):
    df = df_imgs.merge(gt[[id_col, "label"]], left_on="lesion", right_on=id_col, how="inner").reset_index(drop=True)
else:
    df = df_imgs.merge(gt[[id_col, "label"]], left_on="stem", right_on=id_col, how="inner").reset_index(drop=True)

print(f"\n✓ Matched {len(df):,} images across {len(classes)} classes:\n")
counts = [int((df['label'] == i).sum()) for i in range(len(classes))]
for i, c in enumerate(classes):
    pct = (counts[i] / len(df) * 100) if len(df) > 0 else 0
    print(f"   [{i:2d}] {c:12s}: {counts[i]:5d} images ({pct:5.2f}%)")

# Visual Plot: Class Distribution Bar Chart
if len(df) > 0:
    plt.figure(figsize=(11, 4))
    bars = plt.bar(classes, counts, color='teal', edgecolor='black', alpha=0.85)
    plt.xlabel('Skin Lesion Class', fontweight='bold', fontsize=11)
    plt.ylabel('Number of Images', fontweight='bold', fontsize=11)
    plt.title(f'MILK10k Dataset Class Distribution (Total: {len(df)} images)', fontweight='bold', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 15, f'{yval}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

## 🧼 Step 3: Morphological Black-Hat Hair Removal & Data Augmentation Preview

In [ ]:
import cv2
import matplotlib.pyplot as plt

def remove_hair(rgb_img):
    """Black-Hat morphological filter with Telea inpainting to remove hair artifacts."""
    gray = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    blackhat = cv2.GaussianBlur(blackhat, (3, 3), 0)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    cleaned = cv2.inpaint(rgb_img, mask, inpaintRadius=1, flags=cv2.INPAINT_TELEA)
    return cleaned, mask

def augment_image(img):
    """Affine rotation, shifts, shear, and flips."""
    h, w = img.shape[:2]
    angle = np.random.uniform(-10.0, 10.0)
    tx = np.random.uniform(-0.2, 0.2) * w
    ty = np.random.uniform(-0.2, 0.2) * h
    shear = np.random.uniform(-0.2, 0.2)

    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    M[0, 2] += tx
    M[0, 1] += shear
    out = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    if np.random.rand() < 0.5: out = cv2.flip(out, 1)
    if np.random.rand() < 0.5: out = cv2.flip(out, 0)
    return np.ascontiguousarray(out)

# --- Visual Preview Output ---
if len(df) > 0:
    sample_path = df['path'].iloc[0]
    sample_bgr = cv2.imread(sample_path)
    sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)
    cleaned_rgb, hair_mask = remove_hair(sample_rgb)
    aug_sample = augment_image(cleaned_rgb)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(sample_rgb); axes[0].set_title("(a) Original Image", fontweight='bold')
    axes[1].imshow(hair_mask, cmap='gray'); axes[1].set_title("(b) Black-Hat Hair Mask", fontweight='bold')
    axes[2].imshow(cleaned_rgb); axes[2].set_title("(c) Cleaned Lesion", fontweight='bold')
    axes[3].imshow(aug_sample); axes[3].set_title("(d) Augmented Preview", fontweight='bold')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
    print("✓ Step 3 Complete: Preprocessing & Data Augmentation verified on sample!")

## 🏗️ Step 4: Custom ResNet-50 Architecture from GitHub Repository
#### Sourced from [Sudhandar/ResNet-50-model](https://github.com/Sudhandar/ResNet-50-model)
Defines the exact `identity_block` and `convolutional_block`, builds the 50-layer DAG, and injects ImageNet weights.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization,
    Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalAveragePooling2D, Dropout
)
from tensorflow.keras.initializers import glorot_uniform

# --- Exact block code from Sudhandar/ResNet-50-model/main_script.py ---

def identity_block(X, f, filters, stage, block):
    conv_name_base = f'res{stage}{block}_branch'
    bn_name_base = f'bn{stage}{block}_branch'
    F1, F2, F3 = filters
    X_shortcut = X

    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(1, 1), padding='valid',
               name=conv_name_base + '2a', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding='same',
               name=conv_name_base + '2b', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding='valid',
               name=conv_name_base + '2c', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)

    X = Add()([X_shortcut, X])
    X = Activation('relu')(X)
    return X

def convolutional_block(X, f, filters, stage, block, s=2):
    conv_name_base = f'res{stage}{block}_branch'
    bn_name_base = f'bn{stage}{block}_branch'
    F1, F2, F3 = filters
    X_shortcut = X

    X = Conv2D(F1, (1, 1), strides=(s, s), name=conv_name_base + '2a',
               padding='valid', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(F2, (f, f), strides=(1, 1), name=conv_name_base + '2b',
               padding='same', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv2D(F3, (1, 1), strides=(1, 1), name=conv_name_base + '2c',
               padding='valid', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)

    X_shortcut = Conv2D(F3, (1, 1), strides=(s, s), name=conv_name_base + '1',
                        padding='valid', kernel_initializer=glorot_uniform(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=bn_name_base + '1')(X_shortcut)

    X = Add()([X_shortcut, X])
    X = Activation('relu')(X)
    return X

def build_sudhandar_resnet50(input_shape=(224, 224, 3), classes=11):
    X_input = Input(input_shape, name='input_image')
    X = ZeroPadding2D((3, 3), name='conv1_pad')(X_input)

    # Stage 1
    X = Conv2D(64, (7, 7), strides=(2, 2), name='conv1', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name='bn_conv1')(X)
    X = Activation('relu', name='conv1_relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2), padding='same', name='pool1')(X)

    # Stage 2 (3 blocks)
    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, block='a', s=1)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')

    # Stage 3 (4 blocks)
    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')

    # Stage 4 (6 blocks)
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

    # Stage 5 (3 blocks)
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block='a', s=2)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='b')
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='c')

    # Classification Head for MILK10k
    X = GlobalAveragePooling2D(name='gap_head')(X)
    X = Dense(128, activation='relu', name='fc_dense_128')(X)
    X = Dropout(0.3, name='head_dropout')(X)
    X = Dense(classes, activation='softmax', name=f'predictions_{classes}')(X)

    return Model(inputs=X_input, outputs=X, name='Sudhandar_ResNet50')

def transfer_imagenet_weights(custom_model, img_size=224):
    """Ports official pretrained ImageNet weights into Sudhandar's custom architecture layers."""
    official = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(img_size, img_size, 3))
    src_layers = [l for l in official.layers if isinstance(l, (layers.Conv2D, layers.BatchNormalization)) and len(l.weights) > 0]
    dst_layers = [l for l in custom_model.layers if isinstance(l, (layers.Conv2D, layers.BatchNormalization)) and len(l.weights) > 0]
    transferred = 0
    for s, d in zip(src_layers, dst_layers):
        sw, dw = s.get_weights(), d.get_weights()
        if len(sw) == len(dw) and all(a.shape == b.shape for a, b in zip(sw, dw)):
            d.set_weights(sw)
            transferred += 1
    print(f"✓ Loaded ImageNet weights into {transferred} custom layers.")
    del official
    return custom_model

# --- Build and Display Architecture Summary ---
K = len(classes)
test_model = build_sudhandar_resnet50(input_shape=(224, 224, 3), classes=K)
total_params = test_model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in test_model.trainable_weights)

print("=" * 65)
print("✓ Sudhandar ResNet-50 Model Successfully Built & Verified!")
print("=" * 65)
print(f"• Total Network Layers     : {len(test_model.layers)}")
print(f"• Input Tensor Shape       : {test_model.input_shape}")
print(f"• Output Predictions Shape : {test_model.output_shape} ({K} Classes)")
print(f"• Total Parameters         : {total_params:,} ({total_params/1e6:.2f} Million)")
print(f"• Trainable Parameters     : {trainable_params:,}")
print("• Backbone Stages          : 5 Stages (Stage 1 Conv + 16 Residual Blocks)")
print("=" * 65)
del test_model

## 🔄 Step 5: Data Sequence with Dynamic Class Balancing & Batch Verification

In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = 224
BATCH_SIZE = 32
K = len(classes)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

class MILK10kSequence(tf.keras.utils.Sequence):
    def __init__(self, df_part, is_train=True):
        self.df = df_part.reset_index(drop=True)
        self.is_train = is_train
        self.paths = self.df['path'].values
        self.labels = self.df['label'].values.astype(int)
        if self.is_train:
            self.target = int(pd.Series(self.labels).value_counts().max())
        self.on_epoch_end()

    def on_epoch_end(self):
        if not self.is_train:
            self.indices = np.arange(len(self.labels))
            return
        bal = []
        for c in range(K):
            idx = np.where(self.labels == c)[0]
            if len(idx) == 0: continue
            bal.extend(np.random.choice(idx, self.target, replace=True))
        self.indices = np.random.permutation(bal)

    def __len__(self):
        return int(np.ceil(len(self.indices) / BATCH_SIZE))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * BATCH_SIZE : (idx + 1) * BATCH_SIZE]
        imgs = []
        for i in batch_idx:
            im = cv2.imread(self.paths[i])
            if im is None:
                im = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            else:
                im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
                im = cv2.resize(im, (IMG_SIZE, IMG_SIZE))
            if self.is_train:
                im = augment_image(im)
            imgs.append(im)
        X = tf.keras.applications.resnet50.preprocess_input(np.array(imgs, dtype='float32'))
        y = tf.keras.utils.to_categorical(self.labels[batch_idx], num_classes=K)
        return X, y

train_gen = MILK10kSequence(train_df, is_train=True)
val_gen = MILK10kSequence(val_df, is_train=False)

print("=" * 65)
print(f"✓ Training Set   : {len(train_df)} images -> {len(train_gen)} batches/epoch (balanced)")
print(f"✓ Validation Set : {len(val_df)} images -> {len(val_gen)} batches/epoch")
print("=" * 65)

# --- Visual Batch Verification (Plot 4 Batch Images) ---
batch_X, batch_y = train_gen[0]
print(f"✓ Verified Batch 0 Shapes: Images {batch_X.shape} | Labels {batch_y.shape}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    # Denormalize BGR centering for visualization display
    disp_img = batch_X[i].copy()
    disp_img[:, :, 0] += 103.939
    disp_img[:, :, 1] += 116.779
    disp_img[:, :, 2] += 123.68
    disp_img = np.clip(disp_img[:, :, ::-1], 0, 255).astype(np.uint8)
    cls_name = classes[batch_y[i].argmax()]
    axes[i].imshow(disp_img)
    axes[i].set_title(f"Batch #{i}: {cls_name}", fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 🚀 Step 6: Two-Phase Fine-Tuning & Learning Curves Plot
1. **Phase 1 (Warmup)**: Freeze Sudhandar ResNet-50 backbone, train classification head (`lr = 1e-3`, 5 epochs).
2. **Phase 2 (Fine-Tuning)**: Unfreeze residual layers (keep BatchNorm frozen), fine-tune with `lr = 1e-4`, `ReduceLROnPlateau`, and `EarlyStopping`.

In [ ]:
# Instantiate Sudhandar's custom ResNet-50
model = build_sudhandar_resnet50(input_shape=(IMG_SIZE, IMG_SIZE, 3), classes=K)
model = transfer_imagenet_weights(model, img_size=IMG_SIZE)

# =========================================================================
# Phase 1: Train Head Only
# =========================================================================
head_layers = ['gap_head', 'fc_dense_128', 'head_dropout', f'predictions_{K}']
for l in model.layers:
    l.trainable = (l.name in head_layers)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=" * 65)
print("🚀 PHASE 1: Training Classification Head (5 Warmup Epochs)")
print("=" * 65)
history_p1 = model.fit(train_gen, validation_data=val_gen, epochs=5, verbose=1)

# =========================================================================
# Phase 2: Fine-Tune Backbone
# =========================================================================
for l in model.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
    else:
        l.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/best_sudhandar_resnet50.weights.h5", monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)
]

print("\n" + "=" * 65)
print("🚀 PHASE 2: Deep Fine-Tuning ResNet-50 Backbone (25 Epochs)")
print("=" * 65)
history_p2 = model.fit(train_gen, validation_data=val_gen, epochs=25, callbacks=callbacks, verbose=1)

model.save(f"{OUT_DIR}/sudhandar_resnet50_milk10k_final.keras")
print(f"\n✓ Model successfully saved to: {OUT_DIR}/sudhandar_resnet50_milk10k_final.keras")

# --- Visual Plot: Training & Validation Curves ---
acc = history_p1.history.get('accuracy', []) + history_p2.history.get('accuracy', [])
val_acc = history_p1.history.get('val_accuracy', []) + history_p2.history.get('val_accuracy', [])
loss = history_p1.history.get('loss', []) + history_p2.history.get('loss', [])
val_loss = history_p1.history.get('val_loss', []) + history_p2.history.get('val_loss', [])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(acc, label='Training Accuracy', color='royalblue', lw=2)
axes[0].plot(val_acc, label='Validation Accuracy', color='darkorange', lw=2)
axes[0].axvline(x=4.5, color='gray', linestyle='--', label='Phase 2 Unfreeze')
axes[0].set_title('ResNet-50 Accuracy Curve', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].grid(True, linestyle='--', alpha=0.5); axes[0].legend()

axes[1].plot(loss, label='Training Loss', color='royalblue', lw=2)
axes[1].plot(val_loss, label='Validation Loss', color='crimson', lw=2)
axes[1].axvline(x=4.5, color='gray', linestyle='--', label='Phase 2 Unfreeze')
axes[1].set_title('ResNet-50 Loss Curve', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Categorical Crossentropy')
axes[1].grid(True, linestyle='--', alpha=0.5); axes[1].legend()
plt.tight_layout()
plt.show()

## 📈 Step 7: Clinical Evaluation & Confusion Matrix Heatmap

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

# Predict on validation set
print("⏳ Generating predictions for validation dataset...")
val_probs = model.predict(val_gen, verbose=1)
val_preds = val_probs.argmax(axis=1)
val_true = val_df['label'].values[:len(val_preds)]

acc = accuracy_score(val_true, val_preds)
bacc = balanced_accuracy_score(val_true, val_preds)
macro_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(val_true, val_preds, average='weighted', zero_division=0)
class_f1s = f1_score(val_true, val_preds, average=None, zero_division=0)

print("=" * 68)
print(f"📊 Final Test Accuracy      : {acc * 100:.2f}%")
print(f"📊 Final Balanced Accuracy  : {bacc * 100:.2f}%")
print(f"🎯 Overall Macro F1-Score   : {macro_f1:.4f} ({macro_f1 * 100:.2f}%)")
print(f"🎯 Weighted F1-Score        : {weighted_f1:.4f} ({weighted_f1 * 100:.2f}%)")
print("=" * 68)
print("\nDetailed Clinical Classification Report (Precision, Recall, F1):")
print(classification_report(val_true, val_preds, target_names=classes, zero_division=0))

# --- Visual Plot 1: Per-Class F1-Score Bar Chart ---
colors = ['#2ca02c' if f >= 0.8 else ('#ff7f0e' if f >= 0.5 else '#d62728') for f in class_f1s]
plt.figure(figsize=(12, 4.5))
bars = plt.bar(classes, [f * 100 for f in class_f1s], color=colors, edgecolor='black', alpha=0.85)
plt.axhline(y=macro_f1 * 100, color='blue', linestyle='--', linewidth=2, label=f'Macro F1: {macro_f1*100:.2f}%')
plt.xlabel('Skin Lesion Class', fontweight='bold', fontsize=11)
plt.ylabel('F1-Score (%)', fontweight='bold', fontsize=11)
plt.title('Per-Class F1-Score Breakdown (Green >= 80%, Orange 50-80%, Red < 50%)', fontweight='bold', fontsize=12)
plt.ylim(0, 105)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.legend(loc='upper right')
plt.savefig(f'{OUT_DIR}/f1_score_breakdown.png', dpi=300)
plt.show()
print(f'✓ Saved F1-score breakdown chart to: {OUT_DIR}/f1_score_breakdown.png')

# --- Visual Plot 2: Confusion Matrix Heatmap ---
cm = confusion_matrix(val_true, val_preds)
plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(K), classes, rotation=45, ha='right', fontsize=9)
plt.yticks(range(K), classes, fontsize=9)
plt.xlabel('Predicted Class', fontweight='bold', fontsize=11)
plt.ylabel('Actual Class', fontweight='bold', fontsize=11)
plt.title('Confusion Matrix - Fine-Tuned Sudhandar ResNet-50 (MILK10k)', fontweight='bold', fontsize=12)
for i in range(K):
    for j in range(K):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=300)
plt.show()
print(f'✓ Saved Confusion Matrix heatmap to: {OUT_DIR}/confusion_matrix.png')

# --- Save Reports to CSV and JSON ---
rep_dict = classification_report(val_true, val_preds, target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(rep_dict).transpose().to_csv(f'{OUT_DIR}/classification_report.csv')
import json
with open(f'{OUT_DIR}/metrics_summary.json', 'w') as jf:
    json.dump({'test_accuracy': acc, 'balanced_accuracy': bacc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1, 'per_class_f1': dict(zip(classes, [float(x) for x in class_f1s]))}, jf, indent=4)
print(f'✓ Saved classification report CSV and summary JSON to: {OUT_DIR}')

## 🩺 Step 8: Single Image Diagnostic Prediction & Confidence Chart

In [ ]:
def predict_lesion(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cleaned, _ = remove_hair(img_rgb)
    resized = cv2.resize(cleaned, (IMG_SIZE, IMG_SIZE))
    inp = tf.keras.applications.resnet50.preprocess_input(np.expand_dims(resized.astype('float32'), axis=0))
    probs = model.predict(inp, verbose=0)[0]
    top_idx = probs.argmax()
    
    # Plot side-by-side: Image and Horizontal Confidence Bar Chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cleaned)
    axes[0].set_title(f"Diagnosed: {classes[top_idx]} ({probs[top_idx]*100:.1f}%)", fontweight='bold', fontsize=12)
    axes[0].axis('off')
    
    top5_idx = probs.argsort()[::-1][:5]
    top5_names = [classes[i] for i in top5_idx][::-1]
    top5_scores = [probs[i]*100 for i in top5_idx][::-1]
    
    axes[1].barh(top5_names, top5_scores, color='steelblue', edgecolor='black')
    axes[1].set_xlabel('Probability Confidence (%)', fontweight='bold')
    axes[1].set_title('Top 5 Class Confidence Predictions', fontweight='bold', fontsize=12)
    axes[1].set_xlim(0, 100)
    for i, v in enumerate(top5_scores):
        axes[1].text(v + 1, i, f"{v:.2f}%", va='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run diagnostic test on sample validation image
test_sample = val_df['path'].iloc[0]
true_label = classes[val_df['label'].iloc[0]]
print(f"\n🔬 Testing Sample Image : {Path(test_sample).name}")
print(f"🏷️ True Doctor Diagnosis : {true_label}")
predict_lesion(test_sample)

## 💾 Step 9: Download All Validation Matrices & Charts as a ZIP File

In [ ]:
import shutil, os, glob
from google.colab import files

zip_base = '/content/milk10k_validation_matrices'
zip_path = f'{zip_base}.zip'
export_dir = '/content/exported_matrices'
os.makedirs(export_dir, exist_ok=True)

# Copy all metric figures, CSVs, and JSON files into export folder
for f in glob.glob(f'{OUT_DIR}/*.*'):
    if f.lower().endswith(('.png', '.csv', '.json')):
        shutil.copy(f, export_dir)

# Compress into ZIP archive
shutil.make_archive(zip_base, 'zip', export_dir)

print('=' * 68)
print('📦 ALL VALIDATION MATRICES & CHARTS READY FOR DOWNLOAD!')
print('=' * 68)
for item in sorted(os.listdir(export_dir)):
    fsize = os.path.getsize(os.path.join(export_dir, item)) / 1024
    print(f'  • {item:35s} ({fsize:6.1f} KB)')
print('=' * 68)

# Trigger browser download directly to computer
print('\n⬇️ Triggering direct browser download to your computer...')
files.download(zip_path)
print('✓ Download initialized! Check your browser downloads folder.')
